In [1]:
 from google.colab import drive
 drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# %% --- Cell 1: Setup & Mount Drive ---
!pip install -q torch torchvision scikit-learn matplotlib seaborn opencv-python-headless tqdm

import os, random, shutil, numpy as np, cv2, matplotlib.pyplot as plt, seaborn as sns
from PIL import Image
from collections import Counter
from pathlib import Path
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize

print(f"PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available(): print(f"GPU: {torch.cuda.get_device_name(0)}")


PyTorch: 2.10.0+cpu | CUDA: False


In [4]:
# %% --- Cell 2: Config ---
class CONFIG:
    # ===== ĐƯỜNG DẪN — SỬA LẠI THEO DRIVE CỦA BẠN =====
    DRIVE_ROOT    = "/content/drive/MyDrive/SMIDS_28_4/DATA"
    RAW_DATA_DIR  = os.path.join(DRIVE_ROOT, "/content/drive/MyDrive/SMIDS_28_4")       # Chứa 3 thư mục class
    PROCESSED_DIR = os.path.join(DRIVE_ROOT, "data/processed")
    CHECKPOINT_DIR= os.path.join(DRIVE_ROOT, "outputs/checkpoints")
    FIGURES_DIR   = os.path.join(DRIVE_ROOT, "outputs/figures")

    NUM_CLASSES  = 3
    CLASS_NAMES  = ["Normal_Sperm", "Abnormal_Sperm", "Non_Sperm"]
    IMAGE_SIZE   = 224           # ← Khôi phục 224 (ban đầu)
    BATCH_SIZE   = 16
    NUM_WORKERS  = 2
    SEED         = 42

    # Fine-tuning 3 stages (tăng epochs)
    EPOCHS  = [10, 15, 12]      # stage1, stage2, stage3
    LRS     = [1e-3, 1e-4, 1e-5]
    WEIGHT_DECAY = 1e-4
    PATIENCE     = 7             # Tăng patience
    LABEL_SMOOTHING = 0.1        # Giúp generalization

    USE_AMP          = True
    GRAD_ACCUM_STEPS = 2   # effective batch = 32

    # Preprocessing — khôi phục bộ lọc ban đầu
    FILTER_TYPE  = "bilateral"   # "bilateral" (ban đầu) hoặc "gaussian"
    CLAHE_CLIP   = 2.0
    CLAHE_TILE   = (8, 8)        # ← Khôi phục (8,8)

    IMAGENET_MEAN = [0.485, 0.456, 0.406]
    IMAGENET_STD  = [0.229, 0.224, 0.225]

for d in [CONFIG.PROCESSED_DIR, CONFIG.CHECKPOINT_DIR, CONFIG.FIGURES_DIR]:
    os.makedirs(d, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [5]:
# %% --- Cell 3: Seed & EarlyStopping ---
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(CONFIG.SEED)

class EarlyStopping:
    def __init__(self, patience=5, min_delta=1e-4):
        self.patience, self.min_delta = patience, min_delta
        self.counter, self.best_loss, self.early_stop = 0, None, False
    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
                print(f"⛔ Early stopping (patience={self.patience})")
        else:
            self.best_loss = val_loss; self.counter = 0


In [6]:
# %% --- Cell 4: Preprocessing ---
def preprocess_image(image_path, target_size=224):
    """Bilateral/Gaussian Filter → CLAHE → Resize."""
    img = cv2.imread(str(image_path))
    if img is None: raise ValueError(f"Không đọc được: {image_path}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    # Bộ lọc — Bilateral mạnh hơn, giữ biên tốt hơn Gaussian
    if CONFIG.FILTER_TYPE == "bilateral":
        img = cv2.bilateralFilter(img, d=9, sigmaColor=75, sigmaSpace=75)
    else:
        img = cv2.GaussianBlur(img, (3, 3), 0)
    # CLAHE trên kênh L
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=CONFIG.CLAHE_CLIP, tileGridSize=CONFIG.CLAHE_TILE).apply(l)
    img = cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2RGB)
    return cv2.resize(img, (target_size, target_size), interpolation=cv2.INTER_AREA)


In [7]:
# %% --- Cell 4b: Auto-detect Class Folders ---
def detect_class_folders(data_dir):
    """Tự động phát hiện và map tên thư mục class trong dataset."""
    data_path = Path(data_dir)
    if not data_path.exists():
        return {n: n for n in CONFIG.CLASS_NAMES}
    found = sorted([d.name for d in data_path.iterdir() if d.is_dir()])
    if not found:
        return {n: n for n in CONFIG.CLASS_NAMES}
    PATTERNS = {
        "Normal_Sperm":   ["normal_sperm", "normal", "normalsperm"],
        "Abnormal_Sperm": ["abnormal_sperm", "abnormal", "abnormalsperm"],
        "Non_Sperm":      ["non_sperm", "non-sperm", "nonsperm", "non sperm", "nosperm"],
    }
    mapping = {}  # folder_name → standard_class_name
    for folder in found:
        norm = folder.lower().replace("-", "_").replace(" ", "_").strip()
        for std_name, pats in PATTERNS.items():
            if norm == std_name.lower() or norm in pats:
                mapping[folder] = std_name; break
        if folder not in mapping:
            mapping[folder] = folder  # Giữ nguyên nếu không match
    print(f"📂 Thư mục tìm thấy: {found}")
    print(f"🔗 Mapping: {mapping}")
    return mapping

In [8]:
# %% --- Cell 5: Split & Preprocess Dataset ---
def split_and_preprocess(raw_dir, out_dir, size=224, seed=42):
    """Đọc raw → auto-detect folders → preprocess → split 70/15/15 → lưu."""
    folder_map = detect_class_folders(raw_dir)
    images, labels = [], []
    for folder, std_class in folder_map.items():
        d = Path(raw_dir) / folder
        if not d.exists(): print(f"⚠️ Missing: {d}"); continue
        count = 0
        for p in sorted(d.glob("*")):
            if p.suffix.lower() in [".png",".jpg",".jpeg",".bmp",".tif"]:
                images.append(str(p)); labels.append(std_class); count += 1
        print(f"  📁 {folder} → {std_class}: {count} ảnh")
    print(f"\n📊 Tổng: {len(images)} ảnh | {Counter(labels)}")
    # Kiểm tra đủ 3 class
    found_classes = set(labels)
    missing = set(CONFIG.CLASS_NAMES) - found_classes
    if missing:
        print(f"❌ THIẾU CLASS: {missing} — Kiểm tra tên thư mục!")

    X_tr, X_tmp, y_tr, y_tmp = train_test_split(images, labels, test_size=0.3, stratify=labels, random_state=seed)
    X_val, X_te, y_val, y_te = train_test_split(X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=seed)

    for split, (X, y) in {"train":(X_tr,y_tr),"val":(X_val,y_val),"test":(X_te,y_te)}.items():
        print(f"🔄 {split}: {len(X)} ảnh")
        for img_path, lbl in tqdm(zip(X, y), total=len(X)):
            save_dir = Path(out_dir)/split/lbl; save_dir.mkdir(parents=True, exist_ok=True)
            save_path = save_dir / Path(img_path).name
            if save_path.exists(): continue
            try:
                proc = preprocess_image(img_path, size)
                cv2.imwrite(str(save_path), cv2.cvtColor(proc, cv2.COLOR_RGB2BGR))
            except Exception as e: print(f"  ❌ {e}")
    print("✅ Done!")


In [9]:
split_and_preprocess(CONFIG.RAW_DATA_DIR, CONFIG.PROCESSED_DIR, CONFIG.IMAGE_SIZE)

📂 Thư mục tìm thấy: ['.ipynb_checkpoints', 'Abnormal_Sperm', 'DATA', 'Non-Sperm', 'Normal_Sperm']
🔗 Mapping: {'.ipynb_checkpoints': '.ipynb_checkpoints', 'Abnormal_Sperm': 'Abnormal_Sperm', 'DATA': 'DATA', 'Non-Sperm': 'Non_Sperm', 'Normal_Sperm': 'Normal_Sperm'}
  📁 .ipynb_checkpoints → .ipynb_checkpoints: 0 ảnh
  📁 Abnormal_Sperm → Abnormal_Sperm: 1005 ảnh
  📁 DATA → DATA: 0 ảnh
  📁 Non-Sperm → Non_Sperm: 974 ảnh
  📁 Normal_Sperm → Normal_Sperm: 1021 ảnh

📊 Tổng: 3000 ảnh | Counter({'Normal_Sperm': 1021, 'Abnormal_Sperm': 1005, 'Non_Sperm': 974})
🔄 train: 2100 ảnh


  0%|          | 0/2100 [00:00<?, ?it/s]

🔄 val: 450 ảnh


  0%|          | 0/450 [00:00<?, ?it/s]

🔄 test: 450 ảnh


  0%|          | 0/450 [00:00<?, ?it/s]

✅ Done!


In [10]:
# %% --- Cell 6: Augmentation & DataLoader ---
train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(CONFIG.IMAGE_SIZE),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomVerticalFlip(0.5),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1, hue=0.05),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize(CONFIG.IMAGENET_MEAN, CONFIG.IMAGENET_STD),
    transforms.RandomErasing(p=0.15, scale=(0.02, 0.15)),
])
val_transforms = transforms.Compose([
    transforms.Resize((CONFIG.IMAGE_SIZE, CONFIG.IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(CONFIG.IMAGENET_MEAN, CONFIG.IMAGENET_STD),
])

class SpermDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.transform = transform
        self.images, self.labels = [], []
        self.class_to_idx = {n: i for i, n in enumerate(CONFIG.CLASS_NAMES)}
        root_path = Path(root_dir)
        if not root_path.exists():
            print(f"⚠️ Thư mục không tồn tại: {root_dir}")
            return
        # Auto-detect folders và map về class chuẩn
        folder_map = detect_class_folders(root_dir)
        for folder, std_class in folder_map.items():
            d = root_path / folder
            if std_class not in self.class_to_idx:
                print(f"⚠️ '{std_class}' (folder '{folder}') không trong CLASS_NAMES")
                continue
            idx = self.class_to_idx[std_class]
            for p in sorted(d.glob("*")):
                if p.suffix.lower() in [".png",".jpg",".jpeg",".bmp",".tif"]:
                    self.images.append(p); self.labels.append(idx)
        # Report
        lc = Counter(self.labels)
        for name, i in self.class_to_idx.items():
            c = lc.get(i, 0)
            status = "✅" if c > 0 else "❌"
            print(f"   {status} {name}: {c} ảnh")
    def __len__(self): return len(self.images)
    def __getitem__(self, idx):
        img = Image.open(self.images[idx]).convert("RGB")
        if self.transform: img = self.transform(img)
        return img, self.labels[idx]

def create_dataloaders():
    print("\n" + "="*50 + "\n📂 LOADING DATASETS\n" + "="*50)
    train_ds = SpermDataset(f"{CONFIG.PROCESSED_DIR}/train", train_transforms)
    val_ds   = SpermDataset(f"{CONFIG.PROCESSED_DIR}/val",   val_transforms)
    test_ds  = SpermDataset(f"{CONFIG.PROCESSED_DIR}/test",  val_transforms)

    # Validate đủ 3 class
    train_counts = Counter(train_ds.labels)
    missing = [CONFIG.CLASS_NAMES[i] for i in range(CONFIG.NUM_CLASSES)
               if train_counts.get(i, 0) == 0]
    if missing:
        print(f"\n❌❌❌ CRITICAL: Class không có data: {missing}")
        print(f"   → Model KHÔNG THỂ học các class này!")
        print(f"   → Kiểm tra lại tên thư mục trong '{CONFIG.RAW_DATA_DIR}'")

    # WeightedRandomSampler
    total = len(train_ds.labels)
    cw = {c: total/n for c, n in train_counts.items() if n > 0}
    sampler = WeightedRandomSampler(
        [cw.get(l, 1.0) for l in train_ds.labels], num_samples=total
    )

    # Class weights cho loss
    wt = torch.tensor(
        [total / (CONFIG.NUM_CLASSES * max(train_counts.get(i, 0), 1))
         for i in range(CONFIG.NUM_CLASSES)],
        dtype=torch.float32
    )
    wt = wt / wt.sum() * CONFIG.NUM_CLASSES

    kw = dict(num_workers=CONFIG.NUM_WORKERS, pin_memory=True)
    tl = DataLoader(train_ds, batch_size=CONFIG.BATCH_SIZE, sampler=sampler, drop_last=True, **kw)
    vl = DataLoader(val_ds,   batch_size=CONFIG.BATCH_SIZE, shuffle=False, **kw)
    tel= DataLoader(test_ds,  batch_size=CONFIG.BATCH_SIZE, shuffle=False, **kw)
    print(f"\n✅ Train:{len(train_ds)} | Val:{len(val_ds)} | Test:{len(test_ds)}")
    print(f"   Class weights: {wt.numpy()}")
    return tl, vl, tel, wt


In [11]:

# === TẠO DATALOADERS ===
train_loader, val_loader, test_loader, class_weights = create_dataloaders()



📂 LOADING DATASETS
📂 Thư mục tìm thấy: ['Abnormal_Sperm', 'Non_Sperm', 'Normal_Sperm']
🔗 Mapping: {'Abnormal_Sperm': 'Abnormal_Sperm', 'Non_Sperm': 'Non_Sperm', 'Normal_Sperm': 'Normal_Sperm'}
   ✅ Normal_Sperm: 715 ảnh
   ✅ Abnormal_Sperm: 703 ảnh
   ✅ Non_Sperm: 682 ảnh
📂 Thư mục tìm thấy: ['Abnormal_Sperm', 'Non_Sperm', 'Normal_Sperm']
🔗 Mapping: {'Abnormal_Sperm': 'Abnormal_Sperm', 'Non_Sperm': 'Non_Sperm', 'Normal_Sperm': 'Normal_Sperm'}
   ✅ Normal_Sperm: 153 ảnh
   ✅ Abnormal_Sperm: 151 ảnh
   ✅ Non_Sperm: 146 ảnh
📂 Thư mục tìm thấy: ['Abnormal_Sperm', 'Non_Sperm', 'Normal_Sperm']
🔗 Mapping: {'Abnormal_Sperm': 'Abnormal_Sperm', 'Non_Sperm': 'Non_Sperm', 'Normal_Sperm': 'Normal_Sperm'}
   ✅ Normal_Sperm: 153 ảnh
   ✅ Abnormal_Sperm: 151 ảnh
   ✅ Non_Sperm: 146 ảnh

✅ Train:2100 | Val:450 | Test:450
   Class weights: [0.978647  0.9953522 1.026001 ]


In [12]:
# %% --- Cell 7: Model ResNet50 ---
class SpermClassifier(nn.Module):
    def __init__(self, num_classes=3, pretrained=True):
        super().__init__()
        self.backbone = models.resnet50(weights='IMAGENET1K_V2' if pretrained else None)
        for p in self.backbone.parameters(): p.requires_grad = False
        inf = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Dropout(0.4), nn.Linear(inf, 256), nn.BatchNorm1d(256),
            nn.ReLU(True), nn.Dropout(0.2), nn.Linear(256, num_classes))
    def forward(self, x): return self.backbone(x)
    def unfreeze_layer4(self):
        for p in self.backbone.layer4.parameters(): p.requires_grad = True
        print("🔓 Layer4 unfrozen")
    def unfreeze_all(self):
        for p in self.backbone.parameters(): p.requires_grad = True
        print("🔓 All unfrozen")
    def count_params(self):
        t = sum(p.numel() for p in self.parameters())
        tr = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"📊 {tr:,} trainable / {t:,} total ({100*tr/t:.1f}%)")


In [13]:
# %% --- Cell 7b: Khởi tạo Model ---
model = SpermClassifier().to(DEVICE)
model.count_params()

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:01<00:00, 87.8MB/s]


📊 525,827 trainable / 24,033,859 total (2.2%)


In [14]:
# %% --- Cell 8: Training Functions ---
def save_ckpt(model, opt, epoch, vl, va, path):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save({'epoch':epoch,'model':model.state_dict(),'opt':opt.state_dict(),'vl':vl,'va':va}, path)

def load_ckpt(model, opt, path):
    ck = torch.load(path, map_location=DEVICE)
    model.load_state_dict(ck['model'])
    if opt: opt.load_state_dict(ck['opt'])
    print(f"✅ Loaded epoch {ck['epoch']}, val_loss={ck['vl']:.4f}, val_acc={ck['va']:.2f}%")
    return ck['epoch']+1, ck['vl']

def train_one_epoch(model, loader, crit, opt, scaler, accum=2):
    model.train(); rl, cor, tot = 0., 0, 0; opt.zero_grad()
    for i, (imgs, labs) in enumerate(loader):
        imgs, labs = imgs.to(DEVICE), labs.to(DEVICE)
        with torch.cuda.amp.autocast(enabled=CONFIG.USE_AMP):
            out = model(imgs); loss = crit(out, labs) / accum
        scaler.scale(loss).backward()
        if (i+1) % accum == 0 or (i+1) == len(loader):
            scaler.step(opt); scaler.update(); opt.zero_grad()
        rl += loss.item()*accum*imgs.size(0); _, pred = out.max(1)
        tot += labs.size(0); cor += pred.eq(labs).sum().item()
    return rl/tot, 100.*cor/tot

@torch.no_grad()
def validate(model, loader, crit):
    model.eval(); rl, cor, tot = 0., 0, 0
    for imgs, labs in loader:
        imgs, labs = imgs.to(DEVICE), labs.to(DEVICE)
        with torch.cuda.amp.autocast(enabled=CONFIG.USE_AMP):
            out = model(imgs); loss = crit(out, labs)
        rl += loss.item()*imgs.size(0); _, pred = out.max(1)
        tot += labs.size(0); cor += pred.eq(labs).sum().item()
    return rl/tot, 100.*cor/tot

def train_stage(model, tl, vl, crit, lr, epochs, name):
    print(f"\n{'='*55}\n🚀 {name} | LR={lr} | Epochs={epochs}\n{'='*55}")
    model.count_params()
    opt = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr, weight_decay=CONFIG.WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=3, factor=0.5)
    scaler = torch.cuda.amp.GradScaler(enabled=CONFIG.USE_AMP)
    es = EarlyStopping(CONFIG.PATIENCE)
    hist = {'tl':[],'ta':[],'vl':[],'va':[]}; best = float('inf')

    for ep in range(epochs):
        tls, tas = train_one_epoch(model, tl, crit, opt, scaler, CONFIG.GRAD_ACCUM_STEPS)
        vls, vas = validate(model, vl, crit)
        sched.step(vls)
        hist['tl'].append(tls); hist['ta'].append(tas)
        hist['vl'].append(vls); hist['va'].append(vas)
        print(f"  [{ep+1}/{epochs}] TrL:{tls:.4f} TrA:{tas:.1f}% | VaL:{vls:.4f} VaA:{vas:.1f}%")
        save_ckpt(model, opt, ep, vls, vas, f"{CONFIG.CHECKPOINT_DIR}/{name}_latest.pth")
        if vls < best:
            best = vls
            save_ckpt(model, opt, ep, vls, vas, f"{CONFIG.CHECKPOINT_DIR}/{name}_best.pth")
            print(f"  💾 Best saved (val_loss={vls:.4f})")
        es(vls)
        if es.early_stop: break

    bp = f"{CONFIG.CHECKPOINT_DIR}/{name}_best.pth"
    if os.path.exists(bp): load_ckpt(model, None, bp)
    return hist

In [15]:
# %% --- Cell 9: Run Full Training ---
def run_training(model, tl, vl, cw):
    crit = nn.CrossEntropyLoss(weight=cw.to(DEVICE), label_smoothing=CONFIG.LABEL_SMOOTHING)
    stages = [
        ("stage1_head",   CONFIG.LRS[0], CONFIG.EPOCHS[0], lambda: None),
        ("stage2_layer4", CONFIG.LRS[1], CONFIG.EPOCHS[1], model.unfreeze_layer4),
        ("stage3_full",   CONFIG.LRS[2], CONFIG.EPOCHS[2], model.unfreeze_all),
    ]
    history = {}
    for name, lr, ep, unfreeze_fn in stages:
        unfreeze_fn()
        history[name] = train_stage(model, tl, vl, crit, lr, ep, name)
    torch.save(model.state_dict(), f"{CONFIG.CHECKPOINT_DIR}/final_model.pth")
    print(f"\n🏁 Done! Saved: {CONFIG.CHECKPOINT_DIR}/final_model.pth")
    return history


In [ ]:

# === BẮT ĐẦU TRAINING ===
all_history = run_training(model, train_loader, val_loader, class_weights)



🚀 stage1_head | LR=0.001 | Epochs=10
📊 525,827 trainable / 24,033,859 total (2.2%)


/tmp/ipykernel_1751/331199.py:42: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=CONFIG.USE_AMP)
/usr/local/lib/python3.12/dist-packages/torch/cuda/amp/grad_scaler.py:31: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  super().__init__(
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/tmp/ipykernel_1751/331199.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=CONFIG.USE_AMP):
/usr/local/lib/python3.12/dist-packages/torch/cuda/amp/autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.


  [1/10] TrL:0.9060 TrA:62.2% | VaL:0.7653 VaA:71.3%
  💾 Best saved (val_loss=0.7653)
  [2/10] TrL:0.8437 TrA:66.4% | VaL:0.7521 VaA:72.7%
  💾 Best saved (val_loss=0.7521)
  [3/10] TrL:0.8111 TrA:68.3% | VaL:0.7061 VaA:76.7%
  💾 Best saved (val_loss=0.7061)
  [4/10] TrL:0.8060 TrA:68.2% | VaL:0.7124 VaA:74.9%
  [5/10] TrL:0.7635 TrA:72.4% | VaL:0.7460 VaA:70.9%


In [ ]:
# %% --- Cell 10: Plot Training Curves ---
def plot_history(hist):
    fig, ax = plt.subplots(1, 2, figsize=(15, 5))
    colors = {'stage1_head':'#FF6B6B','stage2_layer4':'#4ECDC4','stage3_full':'#45B7D1'}
    off = 0
    for s, h in hist.items():
        ep = range(off, off+len(h['tl']))
        ax[0].plot(ep, h['tl'], '-',  color=colors[s], label=f'{s} train')
        ax[0].plot(ep, h['vl'], '--', color=colors[s], label=f'{s} val')
        ax[1].plot(ep, h['ta'], '-',  color=colors[s], label=f'{s} train')
        ax[1].plot(ep, h['va'], '--', color=colors[s], label=f'{s} val')
        off += len(h['tl'])
    for a, t in zip(ax, ['Loss','Accuracy (%)']):
        a.set_title(t, fontweight='bold'); a.set_xlabel('Epoch'); a.legend(); a.grid(alpha=0.3)
    plt.suptitle('Training History — 3 Stages', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f"{CONFIG.FIGURES_DIR}/training_curves.png", dpi=150, bbox_inches='tight')
    plt.show()

plot_history(all_history)


In [ ]:
# %% --- Cell 11: Evaluate on Test Set ---
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    preds, lbls, probs = [], [], []
    for imgs, labs in tqdm(loader, desc="Testing"):
        imgs = imgs.to(DEVICE)
        with torch.cuda.amp.autocast(enabled=CONFIG.USE_AMP):
            out = model(imgs)
        pr = F.softmax(out, dim=1)
        preds.extend(out.argmax(1).cpu().numpy())
        lbls.extend(labs.numpy()); probs.extend(pr.cpu().numpy())
    preds, lbls, probs = np.array(preds), np.array(lbls), np.array(probs)

    # --- Classification Report ---
    print("\n" + "="*60 + "\n📊 CLASSIFICATION REPORT\n" + "="*60)
    print(classification_report(lbls, preds, target_names=CONFIG.CLASS_NAMES, digits=4))

    # --- Per-class Metrics ---
    cm = confusion_matrix(lbls, preds)
    n_classes = len(CONFIG.CLASS_NAMES)
    overall_acc = 100 * np.mean(preds == lbls)

    print("="*60)
    print("📋 CHI TIẾT METRICS TỪNG LỚP")
    print("="*60)
    print(f"{'Lớp':<20} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'Specificity':>12} {'F1-Score':>10}")
    print("-"*72)

    per_class = {}
    for i, cls in enumerate(CONFIG.CLASS_NAMES):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = cm.sum() - tp - fn - fp

        acc_i   = 100 * (tp + tn) / (tp + tn + fp + fn) if (tp+tn+fp+fn) > 0 else 0
        prec_i  = 100 * tp / (tp + fp) if (tp + fp) > 0 else 0
        rec_i   = 100 * tp / (tp + fn) if (tp + fn) > 0 else 0  # = Sensitivity
        spec_i  = 100 * tn / (tn + fp) if (tn + fp) > 0 else 0
        f1_i    = 2 * prec_i * rec_i / (prec_i + rec_i) if (prec_i + rec_i) > 0 else 0

        per_class[cls] = {
            'accuracy': acc_i, 'precision': prec_i,
            'recall': rec_i, 'specificity': spec_i, 'f1': f1_i,
            'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn
        }
        print(f"{cls:<20} {acc_i:>9.2f}% {prec_i:>9.2f}% {rec_i:>9.2f}% {spec_i:>11.2f}% {f1_i:>9.2f}%")

    # Macro & Weighted averages
    macro_prec = np.mean([v['precision'] for v in per_class.values()])
    macro_rec  = np.mean([v['recall'] for v in per_class.values()])
    macro_f1   = np.mean([v['f1'] for v in per_class.values()])
    macro_spec = np.mean([v['specificity'] for v in per_class.values()])

    print("-"*72)
    print(f"{'Macro Avg':<20} {'':>10} {macro_prec:>9.2f}% {macro_rec:>9.2f}% {macro_spec:>11.2f}% {macro_f1:>9.2f}%")
    print(f"{'Overall Accuracy':<20} {overall_acc:>9.2f}%")
    print("="*60)

    # TP/FP/FN/TN per class
    print("\n📊 CONFUSION DETAILS (TP / FP / FN / TN)")
    print("-"*60)
    for cls, v in per_class.items():
        print(f"  {cls:<20} TP={v['tp']:>4d}  FP={v['fp']:>4d}  FN={v['fn']:>4d}  TN={v['tn']:>4d}")

    return lbls, preds, probs, per_class

y_true, y_pred, y_probs, per_class = evaluate(model, test_loader)


In [ ]:
# %% --- Cell 11b: Per-class Metrics Bar Chart ---
def plot_per_class_metrics(per_class):
    """Biểu đồ bar chart so sánh metrics giữa 3 lớp."""
    metrics = ['accuracy', 'precision', 'recall', 'specificity', 'f1']
    labels  = ['Accuracy', 'Precision', 'Recall\n(Sensitivity)', 'Specificity', 'F1-Score']
    classes = list(per_class.keys())
    colors  = ['#4ECDC4', '#FF6B6B', '#95A5A6']

    x = np.arange(len(metrics))
    width = 0.25

    fig, ax = plt.subplots(figsize=(14, 6))
    for i, (cls, color) in enumerate(zip(classes, colors)):
        vals = [per_class[cls][m] for m in metrics]
        bars = ax.bar(x + i*width, vals, width, label=cls, color=color, edgecolor='white', linewidth=0.5)
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                    f'{v:.1f}%', ha='center', va='bottom', fontsize=8, fontweight='bold')

    ax.set_ylabel('Score (%)', fontsize=12)
    ax.set_title('📊 Per-class Metrics Comparison', fontsize=14, fontweight='bold')
    ax.set_xticks(x + width)
    ax.set_xticklabels(labels, fontsize=11)
    ax.set_ylim(0, 110)
    ax.legend(fontsize=10)
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{CONFIG.FIGURES_DIR}/per_class_metrics.png", dpi=150, bbox_inches='tight')
    plt.show()

plot_per_class_metrics(per_class)


In [ ]:
# %% --- Cell 12: Confusion Matrix ---
def plot_cm(yt, yp):
    cm = confusion_matrix(yt, yp)
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
    fig, ax = plt.subplots(1, 2, figsize=(13, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CONFIG.CLASS_NAMES,
                yticklabels=CONFIG.CLASS_NAMES, ax=ax[0])
    ax[0].set_title('Counts'); ax[0].set_xlabel('Predicted'); ax[0].set_ylabel('Actual')
    sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Oranges', xticklabels=CONFIG.CLASS_NAMES,
                yticklabels=CONFIG.CLASS_NAMES, ax=ax[1])
    ax[1].set_title('Percentage (%)'); ax[1].set_xlabel('Predicted'); ax[1].set_ylabel('Actual')
    plt.tight_layout()
    plt.savefig(f"{CONFIG.FIGURES_DIR}/confusion_matrix.png", dpi=150, bbox_inches='tight')
    plt.show()

plot_cm(y_true, y_pred)


In [ ]:
# %% --- Cell 13: ROC Curves ---
def plot_roc(yt, yp):
    yb = label_binarize(yt, classes=[0,1,2])
    fig, ax = plt.subplots(figsize=(8,6))
    for i, (cls, c) in enumerate(zip(CONFIG.CLASS_NAMES, ['#FF6B6B','#4ECDC4','#45B7D1'])):
        fpr, tpr, _ = roc_curve(yb[:,i], yp[:,i])
        ax.plot(fpr, tpr, color=c, lw=2, label=f'{cls} (AUC={auc(fpr,tpr):.4f})')
    ax.plot([0,1],[0,1],'k--'); ax.set_xlabel('FPR'); ax.set_ylabel('TPR')
    ax.set_title('ROC Curves (OvR)', fontweight='bold'); ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"{CONFIG.FIGURES_DIR}/roc_curves.png", dpi=150, bbox_inches='tight')
    plt.show()

plot_roc(y_true, y_probs)


In [ ]:
# %% --- Cell 14: Misclassified Samples ---
def show_misclassified(model, loader, n=8):
    model.eval()
    wrong_imgs, wrong_p, wrong_l = [], [], []
    mean = torch.tensor(CONFIG.IMAGENET_MEAN).view(3,1,1)
    std  = torch.tensor(CONFIG.IMAGENET_STD).view(3,1,1)
    with torch.no_grad():
        for imgs, labs in loader:
            out = model(imgs.to(DEVICE)); pred = out.argmax(1).cpu()
            mask = pred != labs
            if mask.any():
                wrong_imgs.extend(imgs[mask]); wrong_p.extend(pred[mask].numpy())
                wrong_l.extend(labs[mask].numpy())
            if len(wrong_imgs) >= n: break
    if not wrong_imgs: print("✅ No misclassified!"); return
    k = min(n, len(wrong_imgs)); cols=4; rows=(k+3)//4
    fig, ax = plt.subplots(rows, cols, figsize=(14, 3.5*rows))
    ax = np.array(ax).flatten()
    for i in range(k):
        im = (wrong_imgs[i]*std+mean).permute(1,2,0).numpy().clip(0,1)
        ax[i].imshow(im); ax[i].axis('off')
        ax[i].set_title(f"True:{CONFIG.CLASS_NAMES[wrong_l[i]]}\nPred:{CONFIG.CLASS_NAMES[wrong_p[i]]}", color='red', fontsize=9)
    for j in range(k, len(ax)): ax[j].axis('off')
    plt.suptitle('❌ Misclassified', fontweight='bold'); plt.tight_layout()
    plt.savefig(f"{CONFIG.FIGURES_DIR}/misclassified.png", dpi=150, bbox_inches='tight')
    plt.show()

show_misclassified(model, test_loader)

In [ ]:
# %% --- Cell 15: Inference ---
def predict_image(image_path, model):
    proc = preprocess_image(image_path, CONFIG.IMAGE_SIZE)
    tensor = val_transforms(Image.fromarray(proc)).unsqueeze(0).to(DEVICE)
    model.eval()
    with torch.no_grad():
        with torch.cuda.amp.autocast(enabled=CONFIG.USE_AMP):
            logits = model(tensor)
        probs = F.softmax(logits, dim=1).squeeze().cpu().numpy()
    idx = probs.argmax()
    return CONFIG.CLASS_NAMES[idx], probs[idx], probs

def predict_and_show(image_path, model):
    label, conf, probs = predict_image(image_path, model)
    fig, ax = plt.subplots(1, 2, figsize=(12, 5))
    img = cv2.cvtColor(cv2.imread(str(image_path)), cv2.COLOR_BGR2RGB)
    ax[0].imshow(img); ax[0].set_title(f"{label} ({conf:.1%})", fontsize=13); ax[0].axis('off')
    bars = ax[1].barh(CONFIG.CLASS_NAMES, probs, color=['#4ECDC4','#FF6B6B','#95A5A6'])
    ax[1].set_xlim(0,1); ax[1].set_title('Probabilities', fontweight='bold')
    for b, p in zip(bars, probs): ax[1].text(b.get_width()+0.02, b.get_y()+b.get_height()/2, f'{p:.1%}', va='center')
    plt.tight_layout(); plt.show()
    return label, conf

predict_and_show("/path/to/image.jpg", model)


END